In [1]:
import os
import pandas as pd
from typing import List

In [2]:
def get_dir_names(path: str) -> List[str]:
    """Get names of subdirectories in path.

    Args:
        path: path where to find subdirectories

    Returns: list of names of the subdirectories
    """
    directories = []
    for (dirpath, dirnames, filenames) in os.walk(path):
        directories.extend(dirnames)
        break
    return directories

def get_file_names(path: str) -> List[str]:
    """Get names of files with blocked eos data in directory path.

    Args:
        path: path of eos data

    Returns: list of file names
    """
    files = []
    for (dirpath, dirnames, filenames) in os.walk(path):
        files.extend(filenames)
        break
    return list(f for f in files if f.startswith(f'potential') and f.endswith(f'.csv'))

In [3]:
def parse_lattice_sizes(lattice_name):
    if lattice_name.find('x') != -1:
        b = lattice_name.split('x')
        space_size = int(b[0][:b[0].find('^')])
        time_size = int(b[1])
    else:
        space_size = int(lattice_name[:lattice_name.find('^')])
        time_size = space_size
    return time_size, space_size

def get_potential_info(file_path):
    try:
        df = pd.read_csv(file_path)
    except:
        df = pd.DataFrame()
    if df.empty:
        return df
    else:
        if 'copy' in df.columns:
            copies_num = len(df['copy'].unique())
        else:
            copies_num = 0
        smearing_max = int(df['smearing_step'].max())
        time_size_max = int(df['time_size'].max())
        space_size_max = int(df['space_size'].max())
        return pd.DataFrame({'copies_num': [copies_num], 'smearing_max': [smearing_max], 'time_size_max': [time_size_max], 'space_size_max': [space_size_max]})


def find_potential_result(base_path):
    df = pd.DataFrame()
    result_name = 'observables_distribution'
    lattice_dirs = get_dir_names(base_path)
    data = []
    for lattice in lattice_dirs:
        Nt, Ns = parse_lattice_sizes(lattice)
        beta_dirs = get_dir_names(f'{base_path}/{lattice}')
        for beta in beta_dirs:
            beta_value = float(beta[4:])
            smearing_dirs = get_dir_names(f'{base_path}/{lattice}/{beta}')
            for smearing in smearing_dirs:
                file_names = get_file_names(f'{base_path}/{lattice}/{beta}/{smearing}')
                for file_name in file_names:
                    potential_type = file_name[10:][:-4]
                    file_path = f'{base_path}/{lattice}/{beta}/{smearing}/{file_name}'
                    if os.path.isfile(file_path) and os.stat(file_path).st_size != 0:
                        data.append(get_potential_info(file_path))
                        data[-1]['Nt'] = Nt
                        data[-1]['Ns'] = Ns
                        data[-1]['beta'] = beta_value
                        data[-1]['smearing'] = smearing
                        data[-1]['potential_type'] = potential_type
                        data[-1]['sa_steps'] = 0
                sa_steps_dirs = get_dir_names(f'{base_path}/{lattice}/{beta}/{smearing}')
                for sa_steps in sa_steps_dirs:
                    steps_num = int(sa_steps[6:])
                    copy_num_dirs = get_dir_names(f'{base_path}/{lattice}/{beta}/{smearing}/{sa_steps}')
                    for copy_num in copy_num_dirs:
                        file_names = get_file_names(f'{base_path}/{lattice}/{beta}/{smearing}/{sa_steps}/{copy_num}')
                        for file_name in file_names:
                            potential_type = file_name[10:][:-4]
                            file_path = f'{base_path}/{lattice}/{beta}/{smearing}/{sa_steps}/{copy_num}/{file_name}'
                            if os.path.isfile(file_path) and os.stat(file_path).st_size != 0:
                                data.append(get_potential_info(file_path))
                                data[-1]['Nt'] = Nt
                                data[-1]['Ns'] = Ns
                                data[-1]['beta'] = beta_value
                                data[-1]['smearing'] = smearing
                                data[-1]['potential_type'] = potential_type
                                data[-1]['sa_steps'] = steps_num
    return pd.concat(data)

In [5]:
df = find_potential_result('/home/ilya/soft/lattice/observables/result/potential/wilson_gevp/fundamental/on-axis/su3/gluodynamics')
df = df[(df['sa_steps'] == 0) & (df['Ns'] == 24)]
print(df.to_string())
# df.to_csv(f"../../result/potential/wilson_gevp/data_statistics_su3_gluodynamics.csv", index=False)

   copies_num  smearing_max  time_size_max  space_size_max  Nt  Ns  beta                          smearing potential_type  sa_steps
0           0           101             11              12  24  24   6.0  HYP1_alpha=1_1_0.5_APE_alpha=0.5       original         0
0           1            51             23              24  24  24   6.0  HYP1_alpha=1_1_0.5_APE_alpha=0.5     monopoless         0
0           0           101             11              12  24  24   6.0  HYP0_alpha=1_1_0.5_APE_alpha=0.5       original         0
0         100            31             11              12  24  24   6.0  HYP0_alpha=1_1_0.5_APE_alpha=0.5       monopole         0
0         100            31             11              12  24  24   6.0  HYP0_alpha=1_1_0.5_APE_alpha=0.5        abelian         0
0           1           121             23              24  24  24   6.0  HYP0_alpha=1_1_0.5_APE_alpha=0.5       monopole         0
0           2            31             11              12  24  24   6.0  HY